In [1]:
import fasttext
import pandas as pd
import numpy as np
from numpy.linalg import norm

In [2]:
# 加載 FastText 模型
model = fasttext.load_model(r"C:\Users\nicon\Downloads\cc.zh.300.bin\cc.zh.300.bin")

In [9]:
# 讀取專利草稿構面和專利構面資料
draft_df = pd.read_excel('專利草稿構面.xlsx')
patents_df = pd.read_excel('專利構面.xlsx')

In [10]:
# 欄位名稱列表
columns = ['技術特徵', '專利範圍', '獨立與從屬權利要求', '技術問題與解決方案', '摘要和背景資訊', '結構和功能元素', '描述和附圖', 'IPC或CPC分類']

In [11]:
# 存儲相似度結果的字典
similarity_results = {col: [] for col in columns}

In [12]:
# 計算各欄位的相似度
for col in columns:
    # 處理草稿構面的對應欄位
    draft_text = draft_df[col].iloc[0]
    draft_vector = np.squeeze(np.asarray(model.get_sentence_vector(draft_text)))

    for idx, row in patents_df.iterrows():
        patent_text = row[col]
        patent_vector = np.squeeze(np.asarray(model.get_sentence_vector(str(patent_text))))
        similarity = np.dot(draft_vector, patent_vector) / (norm(draft_vector) * norm(patent_vector))
        similarity_results[col].append((idx, similarity))

In [13]:
# 將各欄位的相似度排序並輸出結果
for col in columns:
    similarity_results[col].sort(key=lambda x: x[1], reverse=True)
    print(f"\n欄位: {col}")
    for idx, similarity in similarity_results[col]:
        print(f"Index: {idx+1}, Similarity: {similarity}")


欄位: 技術特徵
Index: 85, Similarity: 1.0000001192092896
Index: 89, Similarity: 0.8422591090202332
Index: 11, Similarity: 0.826073169708252
Index: 86, Similarity: 0.8240941762924194
Index: 42, Similarity: 0.8219932317733765
Index: 20, Similarity: 0.817505955696106
Index: 21, Similarity: 0.8160840272903442
Index: 31, Similarity: 0.8151072859764099
Index: 15, Similarity: 0.8147947192192078
Index: 17, Similarity: 0.8132900595664978
Index: 62, Similarity: 0.8105239272117615
Index: 79, Similarity: 0.8060797452926636
Index: 54, Similarity: 0.8051913976669312
Index: 81, Similarity: 0.8033928871154785
Index: 34, Similarity: 0.8033273220062256
Index: 100, Similarity: 0.8033052086830139
Index: 14, Similarity: 0.801040768623352
Index: 70, Similarity: 0.8002985715866089
Index: 24, Similarity: 0.7996591925621033
Index: 77, Similarity: 0.7995554208755493
Index: 57, Similarity: 0.799098789691925
Index: 99, Similarity: 0.7982606887817383
Index: 7, Similarity: 0.7981055974960327
Index: 28, Similarity: 0.798

In [14]:
# 將各欄位的相似度排序並輸出結果到 Excel
with pd.ExcelWriter('fasttext各構面2_similarity_results.xlsx') as writer:
    for col in columns:
        similarity_results[col].sort(key=lambda x: x[1], reverse=True)
        similarity_df = pd.DataFrame(similarity_results[col], columns=['Index', 'Similarity'])
        similarity_df.to_excel(writer, sheet_name=col, index=False)

print("相似度結果已輸出至 fasttext_similarity_results.xlsx")

相似度結果已輸出至 fasttext_similarity_results.xlsx
